# 🚀 REAL TEXT MANIPULATION (RTM) - A100 PARALLEL 4-MODEL ABLATION (16,000 ITERS)

> **Đột phá tốc độ trên A100 (Chạy song song 4 Model cùng lúc):**
> * **Tối ưu Vectorized `BlockDCT` + `Crop-First (stride=8)`:** Giảm thời gian xử lý CPU mỗi ảnh từ `86ms` xuống còn **`1.8ms` (nhanh gấp 48 lần)**!
> * **Huấn luyện Song Song 4 Mô Hình (`M1 + M2 + M3 + M4` cùng lúc trên 1 GPU A100):** Mỗi model chỉ dùng ~5GB VRAM $\rightarrow$ **4 model chạy song song chiếm ~20GB VRAM** (vừa vặn hoàn hảo trong 40GB/80GB VRAM của A100, tận dụng 100% Tensor Cores)!
> * **Thời gian hoàn tất cả 4 model 16,000 iters (`batch_size=4`):** Chỉ khoảng **~1 tiếng - 1 tiếng 20 phút cho CẢ 4 MODEL**!

### 🟢 CELL 1: SETUP TOÀN BỘ MÔI TRƯỜNG & DATASET (~2 Phút)

In [3]:
# 1. Kết nối Google Drive
import os, shutil, glob, site
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone mã nguồn RTM mới nhất từ GitHub
%cd /content
!rm -rf TMD
!git clone -b experiment/prototype-contrastive https://github.com/gagaduy/TMD.git
%cd /content/TMD/ASCFormer

# 3. Copy và Giải nén Dataset RealTextMan từ Google Drive sang Colab SSD
os.makedirs('/content/TMD/ASCFormer/data/ttd', exist_ok=True)
drive_zip = '/content/drive/MyDrive/RealTextMan.zip'
if not os.path.exists(drive_zip):
    matches = glob.glob('/content/drive/MyDrive/**/RealTextMan.zip', recursive=True)
    if matches: drive_zip = matches[0]
    else: raise FileNotFoundError('❌ Không tìm thấy RealTextMan.zip trên Google Drive! Hãy kiểm tra lại file.')

print(f'📦 Đang copy {drive_zip} sang Colab SSD...')
shutil.copy(drive_zip, '/content/TMD/ASCFormer/data/ttd/RealTextMan.zip')
print('📂 Đang giải nén dataset...')
!unzip -q /content/TMD/ASCFormer/data/ttd/RealTextMan.zip -d /content/TMD/ASCFormer/data/ttd/
!rm /content/TMD/ASCFormer/data/ttd/RealTextMan.zip

# 4. Cài đặt các thư viện tương thích tuyệt đối (Khắc phục xung đột NumPy & Python 3.13)
print('⚙️ Đang cài đặt thư viện cho A100...')
!pip uninstall -y mmcv mmcv-lite mmengine mmsegmentation timm yapf > /dev/null 2>&1
!pip install -q "numpy<2.0.0" mmcv-lite==2.0.0 mmengine==0.7.0 timm==0.4.12 einops==0.6.1 prettytable

# Vá triệt để lỗi lib2to3 của Python 3.13
for p in site.getsitepackages():
    yapf_dir = os.path.join(p, 'yapf', 'yapflib')
    os.makedirs(yapf_dir, exist_ok=True)
    with open(os.path.join(p, 'yapf', '__init__.py'), 'w') as f: pass
    with open(os.path.join(yapf_dir, '__init__.py'), 'w') as f: pass
    with open(os.path.join(yapf_dir, 'yapf_api.py'), 'w') as f:
        f.write('def FormatCode(text, **kwargs):\n    return text, True\n')

# Đăng ký module ASCFormer vào Python
for p in site.getsitepackages():
    try:
        with open(os.path.join(p, 'ascformer.pth'), 'w') as f:
            f.write('/content/TMD/ASCFormer\n')
        break
    except Exception: pass

%env PYTHONPATH=/content/TMD/ASCFormer:

# Tạo symlinks cho OpenMMLab
!mkdir -p /content/TMD/ASCFormer/mmseg/.mim
!ln -sf /content/TMD/ASCFormer/tools /content/TMD/ASCFormer/mmseg/.mim/tools
!ln -sf /content/TMD/ASCFormer/configs /content/TMD/ASCFormer/mmseg/.mim/configs

import torch
print('=' * 50)
print('✅ SETUP HOÀN TẤT 100% SẴN SÀNG HUẤN LUYỆN!')
if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)} GB')
print('=' * 50)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
Cloning into 'TMD'...
remote: Enumerating objects: 1634, done.
remote: Counting objects: 100% (1634/1634), done.
remote: Compressing objects: 100% (737/737), done.
remote: Total 1634 (delta 950), reused 1570 (delta 886), pack-reused 0 (from 0)
Receiving objects: 100% (1634/1634), 857.04 KiB | 3.15 MiB/s, done.
Resolving deltas: 100% (950/950), done.
/content/TMD/ASCFormer
📦 Đang copy /content/drive/MyDrive/RealTextMan.zip sang Colab SSD...
📂 Đang giải nén dataset...
replace /content/TMD/ASCFormer/data/ttd/RealTextMan/train.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/TMD/ASCFormer/data/ttd/RealTextMan/test.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/TMD/ASCFormer/data/ttd/RealTextMan/val.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
⚙️ Đang cài đặt thư viện cho A100...
env: PYTHONPATH=/content/TMD/ASCFormer:
✅ SETUP 

### 🔴 CELL 2: CHẠY SONG SONG CÙNG LÚC 4 MODEL ABLATION 16K (`M1 + M2 + M3 + M4`) TRÊN A100

> **Cơ chế chạy song song 4 tiến trình (4-Way Parallel Training):**
> 1. Phóng đồng thời cả 4 mô hình (`M1`, `M2`, `M3`, `M4`) lên GPU A100 (~20GB VRAM tổng cộng, 3 CPU workers/model $\times$ 4 = 12 vCPUs).
> 2. Bảng Dashboard theo dõi trực tiếp tiến độ `Iter [xxxx/16000]` và dung lượng VRAM của cả 4 model mỗi 30 giây.
> 3. Model nào cán đích `16,000` iters là tự động sao lưu ngay lập tức `iter_16000.pth` về Google Drive (`/content/drive/MyDrive/RTM_Ablation16k/`).
> 4. Sau khi cả 4 model hoàn tất, tự động chạy quét đa ngưỡng và in ra **Table 2 ($\tau=0.500$)** & **Table 3 ($\tau^*$ cực đại)**!

In [4]:
import os, re, json, time, shutil, subprocess, torch
from prettytable import PrettyTable

%cd /content/TMD
!git pull origin experiment/prototype-contrastive
%cd /content/TMD/ASCFormer

# Dọn sạch các tiến trình train ngầm cũ (nếu vừa bấm Stop và chạy lại Cell 2)
os.system('pkill -9 -f "tools/train.py" || true')
time.sleep(1)

# 0. Tải trước trọng số Pretrained MiT-B0 1 lần để 4 tiến trình song song không tranh chấp file cache
print('📥 Đang tải sẵn Pretrained MiT-B0 vào bộ nhớ đệm...')
_ = torch.hub.load_state_dict_from_url(
    'https://download.openmmlab.com/mmsegmentation/v0.5/pretrain/segformer/mit_b0_20220624-7e0fe6dd.pth',
    map_location='cpu'
)

DRIVE_SAVE_ROOT = '/content/drive/MyDrive/RTM_Ablation16k'
os.makedirs(DRIVE_SAVE_ROOT, exist_ok=True)

ABLATION_MODELS = [
    ('M1_Baseline_2Stream',  'configs/ascformer/ascformer_rtm_b0_ablation16k_m1_baseline.py',  'work_dirs/ablation16k_m1_baseline'),
    ('M2_TriStream_OCR',     'configs/ascformer/ascformer_rtm_b0_ablation16k_m2_tristream.py', 'work_dirs/ablation16k_m2_tristream'),
    ('M3_TriStream_Focused', 'configs/ascformer/ascformer_rtm_b0_ablation16k_m3_focused.py',   'work_dirs/ablation16k_m3_focused'),
    ('M4_TriStream_Proto',   'configs/ascformer/ascformer_rtm_b0_ablation16k_m4_proto.py',     'work_dirs/ablation16k_m4_proto'),
]

SWEEP_THRESHOLDS = [
    '0.020', '0.025', '0.030', '0.035', '0.040', '0.045', '0.050',
    '0.060', '0.080', '0.100', '0.120', '0.150', '0.200', '0.250',
    '0.300', '0.350', '0.400', '0.450', '0.500'
]

# 1. Khởi chạy SONG SONG cả 4 mô hình trên A100
procs = {}
log_files = {}
backed_up = set()

env = os.environ.copy()
env['PYTHONPATH'] = '/content/TMD/ASCFormer:' + env.get('PYTHONPATH', '')
env['OMP_NUM_THREADS'] = '1'
env['MKL_NUM_THREADS'] = '1'

print('\n' + '=' * 90)
print('🔥 KHỞI ĐỘNG CHẾ ĐỘ HUẤN LUYỆN SONG SONG 4 MÔ HÌNH (M1 + M2 + M3 + M4) TRÊN A100!')
print('=' * 90)

t_start = time.time()
for model_name, cfg_path, work_dir in ABLATION_MODELS:
    drive_model_dir = os.path.join(DRIVE_SAVE_ROOT, model_name)
    os.makedirs(drive_model_dir, exist_ok=True)
    drive_ckpt = os.path.join(drive_model_dir, 'iter_16000.pth')
    local_ckpt = os.path.join(work_dir, 'iter_16000.pth')
    os.makedirs(work_dir, exist_ok=True)

    if os.path.exists(drive_ckpt):
        print(f'✅ [{model_name}] Đã có sẵn iter_16000.pth trên Google Drive! Bỏ qua huấn luyện.')
        if not os.path.exists(local_ckpt):
            shutil.copy(drive_ckpt, local_ckpt)
        backed_up.add(model_name)
        continue

    log_path = os.path.join(work_dir, 'stdout_train.log')
    f_log = open(log_path, 'w')
    log_files[model_name] = (f_log, log_path, work_dir, drive_model_dir)
    cmd = ['python', 'tools/train.py', cfg_path, '--work-dir', work_dir]
    procs[model_name] = subprocess.Popen(cmd, stdout=f_log, stderr=subprocess.STDOUT, env=env)
    print(f'🚀 Đã phóng tiến trình song song cho: {model_name} (PID={procs[model_name].pid})')
    time.sleep(3)  # Giãn cách 3s lúc khởi tạo CUDA context

def get_latest_iter_line(log_path):
    if not os.path.exists(log_path):
        return 'Đang khởi tạo...'
    try:
        with open(log_path, 'r', errors='ignore') as f:
            lines = f.readlines()
        for line in reversed(lines):
            if 'Iter(train)' in line:
                m = re.search(r'Iter\(train\)\s*\[\s*(\d+/\d+)\].*?eta:\s*([0-9:]+).*?loss:\s*([0-9.]+)', line)
                if m:
                    return f'Iter [{m.group(1)}] | Loss: {m.group(3)} | ETA: {m.group(2)}'
                return line.strip()[-65:]
    except Exception:
        pass
    return 'Đang nạp dữ liệu...'

# 2. Vòng lặp giám sát tiến độ song song & tự động sao lưu ngay khi từng model hoàn tất
while len(backed_up) < len(ABLATION_MODELS):
    time.sleep(30)
    elapsed = (time.time() - t_start) / 60.0
    try:
        vram_info = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used,memory.total,utilization.gpu', '--format=csv,noheader'],
            text=True
        ).strip()
    except Exception:
        vram_info = 'N/A'

    print(f'\n⏱️ [{elapsed:.1f} phút] A100 Status (VRAM Used / Total / GPU Util): {vram_info}')
    for model_name, cfg_path, work_dir in ABLATION_MODELS:
        if model_name in backed_up:
            print(f'   ✅ {model_name:<22}: HOÀN TẤT 100% (Đã lưu Google Drive)')
            continue
        p = procs[model_name]
        f_log, log_path, w_dir, d_dir = log_files[model_name]
        ret = p.poll()
        status_str = get_latest_iter_line(log_path)
        if ret is None:
            print(f'   🔄 {model_name:<22}: Đang chạy -> {status_str}')
        elif ret == 0:
            f_log.close()
            print(f'   🎉 {model_name:<22}: VỪA CÁN ĐÍCH 16,000 ITERS! Đang sao lưu lên Google Drive...')
            os.system(f'cp -r {w_dir}/* {d_dir}/')
            backed_up.add(model_name)
        else:
            f_log.close()
            print(f'   ❌ {model_name:<22}: Dừng với mã lỗi {ret}! Xem 20 dòng log cuối:')
            os.system(f'tail -n 20 {log_path}')
            backed_up.add(model_name)

print('\n' + '=' * 90)
print('🎯 TẤT CẢ 4 MÔ HÌNH ĐÃ HUẤN LUYỆN XONG! BẮT ĐẦU QUÉT ĐA NGƯỠNG (THRESHOLD SWEEP)...')
print('=' * 90)

# 3. Chạy quét đa ngưỡng cho cả 4 model
for model_name, cfg_path, work_dir in ABLATION_MODELS:
    drive_model_dir = os.path.join(DRIVE_SAVE_ROOT, model_name)
    local_ckpt = os.path.join(work_dir, 'iter_16000.pth')
    sweep_json = os.path.join(drive_model_dir, f'{model_name}_sweep.json')
    if os.path.exists(local_ckpt) and not os.path.exists(sweep_json):
        print(f'\n🔍 Đang quét đa ngưỡng cho {model_name}...')
        thresh_str = ' '.join(SWEEP_THRESHOLDS)
        !python tools/test_threshold_sweep.py {cfg_path} {local_ckpt} --thresholds {thresh_str} --out-json {sweep_json}

# 4. Tổng hợp và in trực tiếp Table 2 (tau = 0.50) & Table 3 (tau* cực đại)
t2 = PrettyTable(['Model', 'Threshold', 'Tampered IoU (%)', 'Precision (%)', 'Recall (%)', 'Dice / F1 (%)', 'mIoU (%)'])
t3 = PrettyTable(['Model', 'Peak Tau (tau*)', 'Peak IoU (%)', 'Peak Dice (%)', 'Peak mIoU (%)'])

for model_name, _, _ in ABLATION_MODELS:
    sweep_json = os.path.join(DRIVE_SAVE_ROOT, model_name, f'{model_name}_sweep.json')
    if os.path.exists(sweep_json):
        with open(sweep_json, 'r') as f:
            data = json.load(f)
        res_list = data['results']
        r05 = [r for r in res_list if abs(r['threshold'] - 0.50) < 1e-4][0]
        t2.add_row([
            model_name, '0.500',
            f"{r05['tampered_iou']:.2f}%", f"{r05['tampered_precision']:.2f}%",
            f"{r05['tampered_recall']:.2f}%", f"{r05['tampered_dice']:.2f}%", f"{r05['m_iou']:.2f}%"
        ])
        r_best = max(res_list, key=lambda x: x['tampered_iou'])
        t3.add_row([
            model_name, f"{r_best['threshold']:.3f}",
            f"{r_best['tampered_iou']:.2f}%", f"{r_best['tampered_dice']:.2f}%", f"{r_best['m_iou']:.2f}%"
        ])

print('\n📌 TABLE 2: NGƯỠNG MẶC ĐỊNH GỐC (tau = 0.500)')
print(t2)
print('\n📌 TABLE 3: NGƯỠNG CỰC ĐẠI HIỆU CHỈNH (tau* PEAK)')
print(t3)

/content/TMD
From https://github.com/gagaduy/TMD
 * branch            experiment/prototype-contrastive -> FETCH_HEAD
Already up to date.
/content/TMD/ASCFormer
📥 Đang tải sẵn Pretrained MiT-B0 vào bộ nhớ đệm...

🔥 KHỞI ĐỘNG CHẾ ĐỘ HUẤN LUYỆN SONG SONG 4 MÔ HÌNH (M1 + M2 + M3 + M4) TRÊN A100!
🚀 Đã phóng tiến trình song song cho: M1_Baseline_2Stream (PID=15878)
🚀 Đã phóng tiến trình song song cho: M2_TriStream_OCR (PID=15893)
🚀 Đã phóng tiến trình song song cho: M3_TriStream_Focused (PID=15916)
🚀 Đã phóng tiến trình song song cho: M4_TriStream_Proto (PID=15951)

⏱️ [0.7 phút] A100 Status (VRAM Used / Total / GPU Util): 19567 MiB, 40960 MiB, 99 %
   🔄 M1_Baseline_2Stream   : Đang chạy -> Đang nạp dữ liệu...
   🔄 M2_TriStream_OCR      : Đang chạy -> Đang nạp dữ liệu...
   🔄 M3_TriStream_Focused  : Đang chạy -> Đang nạp dữ liệu...
   🔄 M4_TriStream_Proto    : Đang chạy -> Đang nạp dữ liệu...

⏱️ [1.2 phút] A100 Status (VRAM Used / Total / GPU Util): 19669 MiB, 40960 MiB, 99 %
   🔄 M1_Baseli